# G2 — ViT 적용 bridge (payoff 재현 / DeiT-Tiny / CIFAR100 / W3)

G1(CNN)과 **병렬 Colab 세션**으로. ViT는 인프라가 달라 self-contained:
- FP = DeiT-Tiny **pretrained + AdamW finetune** (224 upsample) — `train_baseline`(ResNet SGD) 못 씀
- 프로토콜은 *light* (224라 단일층 full sweep 느림): **PTQ gap + normHd2 proxy + subset payoff(⑤)만** (단일층 진단①②는 생략)
- quant 엔진은 Linear 양자화 됨(`ptq`가 Conv+Linear swap) → proxy/payoff는 generic 엔진 그대로

= "ViT서도 partial QAT payoff 되나" *적용* bridge. **현상 재현이지 recipe 제안 아님. 진단 일반화는 CNN(G1)까지, ViT는 payoff 신호만.** (FP 품질·224 속도·W3 붕괴가 리스크.)

In [1]:
# --- Colab 셋업 ---
import os
REPO = '/content/26_Capstone'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

torch 2.11.0+cu128 | cuda True


In [2]:
# --- 엔진 + Drive + 경로 ---
from qat_engine import *
import numpy as np, json, timm
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

try:
    from google.colab import drive; drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'
os.makedirs(f'{ART}/checkpoints', exist_ok=True); os.makedirs(f'{ART}/outputs/G1', exist_ok=True)
N_BITS = 3
print('device', DEVICE)

Mounted at /content/drive
device cuda


In [3]:
# --- ViT FP: DeiT-Tiny pretrained + AdamW finetune (224 upsample CIFAR100, 캐시) ---
IMG = 224
imnorm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))   # ImageNet norm (pretrained 일관)
tf_tr = transforms.Compose([transforms.Resize(IMG), transforms.RandomCrop(IMG, padding=16),
                            transforms.RandomHorizontalFlip(), transforms.ToTensor(), imnorm])
tf_te = transforms.Compose([transforms.Resize(IMG), transforms.ToTensor(), imnorm])
DR = f'{ART}/data'
tr_set = datasets.CIFAR100(DR, train=True,  download=True, transform=tf_tr)
te_set = datasets.CIFAR100(DR, train=False, download=True, transform=tf_te)
ca_set = Subset(datasets.CIFAR100(DR, train=True, download=True, transform=tf_te), list(range(512)))
train224 = DataLoader(tr_set, batch_size=128, shuffle=True,  num_workers=2, drop_last=True)
val224   = DataLoader(te_set, batch_size=128, shuffle=False, num_workers=2)
calib224 = DataLoader(ca_set, batch_size=64,  shuffle=False, num_workers=2)

VCKPT = f'{ART}/checkpoints/deit_tiny_cifar100_fp.pt'
vit = timm.create_model('deit_tiny_patch16_224', pretrained=True, num_classes=100).to(DEVICE)
if os.path.exists(VCKPT):
    vit.load_state_dict(torch.load(VCKPT, map_location='cpu')); vit = vit.to(DEVICE)
    print('ViT FP 캐시 로드')
else:
    EPOCHS = 5                       # 느리면 3으로. pretrained라 빨리 올라옴
    print(f'ViT finetune (AdamW, {EPOCHS}ep, 224)...')
    opt = torch.optim.AdamW(vit.parameters(), lr=1e-4, weight_decay=0.05)
    crit = nn.CrossEntropyLoss()
    for ep in range(EPOCHS):
        vit.train()
        for x, y in train224:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(vit(x), y); loss.backward(); opt.step()
        a, _ = evaluate_full(vit, val224, DEVICE); print(f'  ep{ep} val acc {a:.2f}')
    torch.save(vit.state_dict(), VCKPT)
vit.eval()
fp_acc, _ = evaluate_full(vit, val224, DEVICE)
print(f'ViT FP acc {fp_acc:.2f}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

ViT FP 캐시 로드
ViT FP acc 80.94


In [4]:
# --- PTQ gap + proxy(normHd2) ---  (엔진 generic: Linear도 양자화/proxy 됨)
# ViT 필수: timm fused SDPA(F.scaled_dot_product_attention)는 이중 backward 미지원 →
#   HVP(‖Hδ‖² proxy)가 'efficient_attention_backward not implemented'로 죽음. 분해가능 attention으로 전환.
_noff = 0
for _m in vit.modules():
    if hasattr(_m, 'fused_attn'):
        _m.fused_attn = False; _noff += 1
print(f'fused_attn off: {_noff} attn modules (HVP 이중backward 가능 / 수치 동일)')
qm = make_ptq_model(vit, N_BITS, DEVICE)
ptq_acc, L_PTQ = evaluate_full(qm, val224, DEVICE)
layers = list_quant_layers(qm)
costs = get_layer_costs(qm, layers)
scores = proxy_scores(qm, vit, N_BITS, calib224, layers=layers, n_batches=8)  # 8배치(=512 calib 전부)로 추정 (PROTO-8)
print(f'ViT: {len(layers)} quant layers | FP {fp_acc:.2f} -> PTQ {ptq_acc:.2f} (gap {fp_acc-ptq_acc:.2f})')
if ptq_acc < max(5.0, 0.10 * fp_acc):   # W3 ViT 붕괴 가드 (VIT-2): recov% 분모(full-ptq)가 noise면 표 해석 불가
    print(f'  [W3-ViT-COLLAPSE] ptq_acc={ptq_acc:.2f} 매우 낮음 → recov% 분모 붕괴 위험.')
    print(f'    권고: cell-2 N_BITS=4로 재실행(W4 ViT 훨씬 안정), 또는 발표선 recov% 대신 절대 acc·loss-R로 보고.')

fused_attn off: 12 attn modules (HVP 이중backward 가능 / 수치 동일)
ViT: 50 quant layers | FP 80.94 -> PTQ 17.04 (gap 63.90)


In [5]:
# --- subset payoff (normHd2-topk vs random×3 vs full) — light (seed2, t<=300) ---
B = 0.25; LR = 1e-3      # ViT recovery 신호 약하면(아래 ⚠ 경고) 1e-2~3e-2로 올려 재실행 (G2=payoff 재현 전용, CNN 동역학 주장 무관)
rng = np.random.default_rng(0)
rules = {'PTQ-only': [], 'full': list(layers),
         'normHd2-topk': select_subset(scores, costs, B, by='normHd2')}
for ri in range(3):       # random은 U5와 동일하게 ×3 평균(단일 draw 분산 회피, HON-2)
    rules[f'random_{ri}'] = select_subset({n: float(rng.random()) for n in layers}, costs, B)
for r, ls in rules.items():
    pr = sum(get_layer_costs(vit, ls).values()) / sum(costs.values()) if ls else 0.0
    print(f'  {r:<14} k={len(ls):<3} param={pr*100:5.1f}%')
u5 = run_u5_subset(vit, N_BITS, rules, train224, val224, steps=300, eval_at=(30, 100, 300), seeds=(0, 1), lr=LR, device=DEVICE)
_rk = [r for r in u5['R'] if r.startswith('random_')]                 # random(avg) 합성
for _d in ('R', 'R_std', 'acc'):
    u5[_d]['random(avg)'] = {t: float(np.mean([u5[_d][r][t] for r in _rk])) for t in u5['t_evals']}
for _d in ('param_ratio', 'wall', 'vram'):
    u5[_d]['random(avg)'] = float(np.mean([u5[_d][r] for r in _rk]))
_T = u5['t_evals'][-1]; _fa = u5['acc']['full'][_T]                    # ViT recovery 신호 사전점검 (VIT-1)
if _fa - ptq_acc < max(3.0, 0.3 * (fp_acc - ptq_acc)):
    print(f'  ⚠ [ViT recovery 약함] full@{_T} acc={_fa:.2f} vs ptq={ptq_acc:.2f} (Δ={_fa-ptq_acc:+.2f})')
    print(f'    → SGD가 ViT를 거의 못 움직임. 위 LR=1e-2~3e-2로 올려 재실행 권장(payoff 신호 확보).')
print('payoff done, L_PTQ', round(u5['L_PTQ'], 4))

  PTQ-only       k=0   param=  0.0%
  full           k=50  param=100.0%
  normHd2-topk   k=11  param= 24.9%
  random_0       k=14  param= 24.6%
  random_1       k=11  param= 24.9%
  random_2       k=17  param= 24.6%
payoff done, L_PTQ 3.8442


In [6]:
# --- 표 + 저장 ---
T = u5['t_evals'][-1]
def recov(a, p, f):
    return 100.0 * (a - p) / (f - p) if f > p else float('nan')
fl = u5['acc']['full'][T]
print(f'DeiT-Tiny / W{N_BITS}   (FP {fp_acc:.2f}, PTQ {ptq_acc:.2f}, full-ptq gap {fl-ptq_acc:+.2f})')
print(f'  rule           acc@{T}   param%   acc-recov%')
for r in ['PTQ-only', 'full', 'normHd2-topk', 'random(avg)']:
    a = u5['acc'][r][T]; pr = u5['param_ratio'][r] * 100
    print(f'  {r:<13} {a:6.2f}   {pr:5.1f}   {recov(a, ptq_acc, fl):6.1f}')
out = dict(model='deit_tiny', fp_acc=float(fp_acc), ptq_acc=float(ptq_acc), gap=float(fp_acc - ptq_acc),
           n_layers=len(layers), B=B, payoff=u5, subsets={r: rules[r] for r in rules})
json.dump(out, open(f'{ART}/outputs/G1/g2_vit_bridge_w{N_BITS}.json', 'w'), indent=2)
print('saved g2_vit_bridge')

DeiT-Tiny / W3   (FP 80.94, PTQ 17.04, full-ptq gap +53.80)
  rule           acc@300   param%   acc-recov%
  PTQ-only       17.04     0.0      0.0
  full           70.84   100.0    100.0
  normHd2-topk   65.28    24.9     89.6
  random(avg)    60.98    24.7     81.7
saved g2_vit_bridge


## G2 해석 (ViT 적용 bridge)

- **`normHd2-topk`가 적은 param%로 full에 근접** → ViT서도 partial QAT payoff 재현 → 발표 **"ViT 적용 bridge: payoff 재현 *신호*"** (진단 일반화는 CNN까지, ViT는 적용⑤만).
- **`normHd2-topk` > `rand(avg)`?** → ViT 단계서도 단기proxy가 *약하게라도* 도움 되는지 (U5와 같은 질문, random은 ×3 평균).

⚠️ 한계(정직):
- **단일층 진단(normHd2→short 상관·역전) 통째로 생략** — 224라 ~50 Linear층 sweep이 너무 느림. G2는 *적용 bridge*지 full 진단 아님(그래서 "확장 *시작*"이지 G1급 "일반화" 아님).
- **W3 per-channel PTQ가 ViT Linear서 붕괴할 수 있음** — cell-4 `ptq_acc`가 한 자릿수면 recov% 분모(full-ptq)가 noise. 그땐 N_BITS=4로 후퇴(주장 무손상) 또는 절대 acc·loss-R로 보고.
- **recovery는 SGD momentum-0**(연구 프로토콜 일관)이라 ViT엔 약할 수 있음 — cell-5 사전점검(full이 ptq 대비 안 오르면)에서 잡고 LR↑로 대응.
- **FP = pretrained finetune** (CIFAR-100 from-scratch ViT는 어려움). FP acc가 낮으면 bridge 약함. ImageNet norm 사용(pretrained 일관).

→ 발표선 **"ViT 적용 bridge: payoff 재현 신호"**. full ViT 진단은 후속(ViT-튜닝 학습 + 경량 단일층).

*트림: 느리면 cell3 `EPOCHS=3`, cell5 `seeds=(0,)`·`steps=100`. (느리다고 LR은 낮추지 말 것 — ViT는 오히려 LR↑가 필요.)*